# 5.2 การจัดสรรแบบมีข้อจำกัด และการจำลอง

แผนทั้งหมดสร้างด้วย `python -m chapter5.build --plans` (ขนานหลายกระบวนการ) และ `--uncertainty`
สมุดบันทึกนี้ (1) แก้ปัญหาแผน P-R ใหม่หนึ่งแผนให้เห็นขั้นตอน และตรวจว่าตรงกับแผนที่เก็บไว้ (2) แสดงสมุดบัญชีรายจังหวัด-ปี (3) แสดงผลภายใต้ความไม่แน่นอน

**ตัวแปรตัดสินใจ:** รับใหม่ $x_{pt}$ และโยกย้าย $z_{pt}$ เป็นจำนวนเต็ม
**สมการคงอยู่:** $H_{pt}=(1-d_p+r_p)H_{p,t-1}+x_{pt}+z^{in}_{pt}-z^{out}_{pt}$
**เป้าหมายหลัก:** $J=\sum_t\sum_p (T_{pt}/\sum_q T_{qt})\,s_{pt}^2,\; s\ge 1-H/T$ แล้วจึงลดความไม่สมดุลของการกระจาย และลดการโยกย้าย ภายในความคลาดเคลื่อน 0.05%

In [1]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "chapter5.toml").exists())
sys.path.insert(0, str(_root / "src"))
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image
pd.options.display.max_columns = 40
pd.options.display.width = 200
from chapter5 import config as C, pipeline as PL, allocation as AL, simulation as SIM, responsibilities as R
S = C.load()
D = S.dirs()
tab = lambda n, f="tables": pd.read_csv(D[f] / f"{n}.csv", encoding="utf-8-sig")
print("scenario:", S.name, "| config_hash:", S.hash())

scenario: canonical | config_hash: e0082b9acf0428bc


In [2]:
ctx = PL.context('{}', '{}')
res = {r['job_id']: r for r in PL.load_plan_results(S)}
job = [j for j in PL.catalogue(S, ('principal',)) if j['job_id'] == 'P-R'][0]
new = PL.run_job(job)
print(new['status'], [ (s['stage'], round(s['objective'], 5), s.get('outcome')) for s in new['stages']])
print('identical to the cached plan:', (new['x'] == res['P-R']['x']).all())
print('constraint check:', new['check'])

exploratory_policy_scenario [('shortfall', 4.35068, 'optimal within gap'), ('distribution', 1.08933, 'optimal within gap')]
identical to the cached plan: True
constraint check: {'pool_conservation': True, 'integer': True, 'floor': True, 'nonnegative_stock': True, 'transfer_balance': True, 'cap': True, 'all_ok': True, 'no_simultaneous_in_and_out': True}


## แผนทั้งหมดและสถานะ

In [3]:
display(tab('T5_PLAN_INDEX', 'validation')[['plan_id', 'group', 'status', 'cap_mode', 'all_constraints_ok', 'max_mip_gap', 'transfer_events', 'stock_2040', 'stage_outcomes']])

,plan_id,group,status,cap_mode,all_constraints_ok,max_mip_gap,transfer_events,stock_2040,stage_outcomes
0,NFB0-R,variant,exploratory_policy_scenario,gross_arrivals,True,NaN,0.0,29974.887136,NaN
1,NFB0.5-R,variant,exploratory_policy_scenario,gross_arrivals,True,NaN,0.0,29951.501855,NaN
2,NFB1-R,variant,exploratory_policy_scenario,gross_arrivals,True,NaN,0.0,29922.874330,NaN
3,NFB1-R|ols,variant,blocked_for_specific_reason,NaN,NaN,NaN,NaN,NaN,NaN
4,NFB1-R|B,variant,exploratory_policy_scenario,gross_arrivals,True,NaN,0.0,29931.238131,NaN
...,...,...,...,...,...,...,...,...,...
84,N-RT|net|tau0.05,sensitivity,exploratory_policy_scenario,net_growth,True,0.000064,0.0,29834.825780,shortfall: optimal within gap; distribution: n...
85,P-RT|net|surplus,sensitivity,exploratory_policy_scenario,net_growth,True,0.004637,0.0,29731.763185,shortfall: time limit: best incumbent; recruit...
86,P-RT|net|region,sensitivity,exploratory_policy_scenario,net_growth,True,0.000067,60.0,29813.671547,shortfall: optimal within gap; distribution: n...
87,B-RT|net,sensitivity,exploratory_policy_scenario,net_growth,True,0.000182,1.0,29746.137857,shortfall: optimal within gap; distribution: t...


## สมุดบัญชีรายจังหวัด-ปี (แผน N-R ปี 2569)

In [4]:
L = res['N-R']['ledger']
cols = ['province', 'opening_headcount', 'expected_resignations', 'expected_retirements', 'expected_departures_total', 'expected_reentries', 'new_appointments_accepted_starts', 'expected_closing_stock', 'target_own', 'attainment_own', 'cap_value', 'cap_binding', 'allocation_reason']
display(L[L.year_ce == 2026][cols].sort_values('attainment_own').head(20))

,province,opening_headcount,expected_resignations,expected_retirements,expected_departures_total,expected_reentries,new_appointments_accepted_starts,expected_closing_stock,target_own,attainment_own,cap_value,cap_binding,allocation_reason
25,บึงกาฬ,72.0,8.978419,0.489473,9.619392,0.358593,6,68.739201,426.606606,0.161130,6.0,True,onboarding cap binds (allocation limited by th...
26,หนองบัวลำภู,113.0,9.606741,0.758872,10.595944,0.596078,10,113.000134,501.648541,0.225258,10.0,True,onboarding cap binds (allocation limited by th...
0,สมุทรปราการ,291.0,19.516185,2.294722,22.635568,2.093606,27,297.458038,1283.061104,0.231835,27.0,True,onboarding cap binds (allocation limited by th...
2,ปทุมธานี,279.0,12.951797,2.140294,15.877571,1.525535,26,290.647964,1151.154523,0.252484,26.0,True,onboarding cap binds (allocation limited by th...
16,สระแก้ว,154.0,17.685132,1.053282,19.571476,0.674247,14,149.102771,522.783253,0.285210,14.0,True,onboarding cap binds (allocation limited by th...
24,อำนาจเจริญ,103.0,5.745414,0.706493,6.666976,0.384100,9,105.717124,369.593583,0.286036,9.0,True,onboarding cap binds (allocation limited by th...
33,กาฬสินธุ์,292.0,13.411970,1.920153,16.144442,1.126294,28,304.981852,1050.450494,0.290334,28.0,True,onboarding cap binds (allocation limited by th...
15,นครนายก,76.0,5.715540,0.551135,6.431593,0.243510,7,76.811917,254.574272,0.301727,7.0,True,onboarding cap binds (allocation limited by th...
48,กำแพงเพชร,209.0,8.469597,1.341196,10.278530,0.704020,20,219.425491,716.414436,0.306283,20.0,True,onboarding cap binds (allocation limited by th...
53,เพชรบูรณ์,283.0,18.696231,1.878617,21.267051,0.845384,27,289.578332,943.240963,0.307004,27.0,True,onboarding cap binds (allocation limited by th...


## ดุลระดับประเทศ (ต้องปิดยอดเป็นศูนย์)

In [5]:
nb_ = pd.read_csv(D['plans'] / 'chapter5_national_annual_balance.csv', encoding='utf-8-sig')
display(nb_[nb_.plan_id.isin(['P-R', 'N-RT'])])
print('max reconciliation difference:', nb_.reconciliation_difference.abs().max())

,plan_id,year_ce,opening_stock,new_appointments,transfers_counted_once,transfers_in,expected_departures,expected_resignations,expected_retirements,expected_other_departures,expected_reentries,closing_stock,target_own,population_civil,plan_status,available,unplaced_balance,reconciliation_difference,transfer_balance_difference,national_attainment_own,year_be
15,N-RT,2026,22685.000000,2000,0,0,1418.379582,1192.785695,166.017225,59.576663,107.512034,23374.132452,59197.545324,5.919755e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.394850,2569
16,N-RT,2027,23374.132452,2000,0,0,1454.645186,1222.366127,170.961078,61.317980,110.416622,24029.903889,59061.645768,5.906165e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.406861,2570
17,N-RT,2028,24029.903889,2000,0,0,1490.466999,1251.846660,175.608562,63.011777,113.364225,24652.801115,58906.336845,5.890634e+07,exploratory_policy_scenario,2000.0,0.0,-3.637979e-12,0,0.418508,2571
18,N-RT,2029,24652.801115,2000,0,0,1523.759986,1279.203241,179.960437,64.596308,116.070954,25245.112082,58731.302436,5.873130e+07,exploratory_policy_scenario,2000.0,0.0,3.637979e-12,0,0.429841,2572
19,N-RT,2030,25245.112082,2000,0,0,1556.614884,1306.308708,184.168693,66.137482,118.808582,25807.305780,58535.719510,5.853572e+07,exploratory_policy_scenario,2000.0,0.0,-3.637979e-12,0,0.440881,2573
20,N-RT,2031,25807.305780,2000,0,0,1587.094449,1331.405528,188.149315,67.539606,121.459763,26341.671094,58319.371353,5.831937e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.451680,2574
21,N-RT,2032,26341.671094,2000,0,0,1617.668442,1356.855485,191.923421,68.889536,123.977062,26847.979715,58081.603211,5.808160e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.462246,2575
22,N-RT,2033,26847.979715,2000,0,0,1647.669146,1381.855585,195.496774,70.316787,126.303580,27326.614149,57822.399783,5.782240e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.472596,2576
23,N-RT,2034,27326.614149,2000,0,0,1676.251303,1405.742604,198.925966,71.582732,128.640778,27779.003624,57542.257740,5.754226e+07,exploratory_policy_scenario,2000.0,0.0,-3.637979e-12,0,0.482758,2577
24,N-RT,2035,27779.003624,2000,0,0,1702.468417,1427.424663,202.233618,72.810136,130.863058,28207.398265,57240.652896,5.724065e+07,exploratory_policy_scenario,2000.0,0.0,0.000000e+00,0,0.492786,2578


max reconciliation difference: 3.637978807091713e-12


## การทดลองป้อนกลับ N-FB

In [6]:
display(tab('T5_FEEDBACK_LOG'))
display(tab('T5_FEEDBACK_CONVERGENCE').query('year_ce in [2026, 2033, 2040]'))

,year,eta_mean,eta_min,eta_max,gain_B_mean,gain_B_max,gain_A_mean,provinces_gain_B_ge_1,plan_id,lam
0,2026,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NFB0-R,0.0
1,2027,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NFB0-R,0.0
2,2028,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NFB0-R,0.0
3,2029,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NFB0-R,0.0
4,2030,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,NFB0-R,0.0
...,...,...,...,...,...,...,...,...,...,...
70,2036,0.307651,0.253989,0.352378,0.621364,1.453422,0.609974,1,NFB1-R|grow,1.0
71,2037,0.304088,0.247427,0.351383,0.602014,1.456380,0.590986,1,NFB1-R|grow,1.0
72,2038,0.301176,0.241593,0.351612,0.584804,1.456290,0.574091,1,NFB1-R|grow,1.0
73,2039,0.297695,0.235799,0.351536,0.567225,1.452807,0.556829,1,NFB1-R|grow,1.0


,plan_id,year_ce,gini_vs_nstd,bottom_quartile_vs_nstd,share_receipts_top19_2025,own_target_total,own_attainment,own_gini
0,N-R,2026,0.115115,0.299711,0.2050,59197.545324,0.394850,0.115115
7,N-R,2033,0.092174,0.350491,0.1830,57822.399783,0.472596,0.092174
14,N-R,2040,0.073512,0.395231,0.2155,55406.781683,0.541633,0.073512
15,NFB0-R,2026,0.115115,0.299711,0.2050,59197.545324,0.394850,0.115115
22,NFB0-R,2033,0.090846,0.350491,0.1800,57822.399783,0.472506,0.090846
29,NFB0-R,2040,0.077001,0.395231,0.1525,55406.781683,0.540997,0.077001
30,NFB0.5-R,2026,0.115058,0.299711,0.2050,59197.545324,0.394850,0.087554
37,NFB0.5-R,2033,0.099350,0.350491,0.2020,57822.399783,0.472504,0.068782
44,NFB0.5-R,2040,0.088054,0.399560,0.1410,55406.781683,0.540575,0.057906
45,NFB1-R,2026,0.117012,0.299711,0.2275,59197.545324,0.394850,0.069023


## ความไม่แน่นอน: แผนตายตัว การปรับแผนรายปี และขอบเขตรู้อนาคต

In [7]:
u = tab('T5_UNCERTAINTY_SUMMARY')
display(u[u.metric.isin(['bottom_quartile_2040', 'cumulative_j'])])
display(tab('T5_UNCERTAINTY_OPERATIONS'))
display(tab('T5_PAIRED_DIFFERENCES', 'validation'))

,plan_id,mode,criterion,metric,mean,p05,p50,p95,n_valid
1,B-R,fixed_schedule,B,bottom_quartile_2040,0.400768,0.381598,0.400866,0.419638,500
4,B-R,fixed_schedule,B,cumulative_j,4.346198,4.206873,4.349057,4.481617,500
7,B-R,fixed_schedule,N,bottom_quartile_2040,0.379309,0.355616,0.379818,0.401413,486
10,B-R,fixed_schedule,N,cumulative_j,4.377978,4.240416,4.380782,4.516240,486
13,B-R,fixed_schedule,P,bottom_quartile_2040,0.369638,0.353082,0.369325,0.387993,500
...,...,...,...,...,...,...,...,...,...
280,P-R,fixed_schedule_same_paths,P,cumulative_j,4.330301,4.198922,4.342965,4.459702,60
283,S-R,fixed_schedule_same_paths,S,bottom_quartile_2040,0.412192,0.396450,0.411266,0.433340,60
286,S-R,fixed_schedule_same_paths,S,cumulative_j,4.273164,4.143024,4.288367,4.401361,60
289,N-R,fixed_schedule_same_paths,N,bottom_quartile_2040,0.387053,0.367109,0.385878,0.410239,60


,plan_id,cancelled_mean,cap_breach_province_years_mean,share_paths_any_cap_breach,protection_breach_mean
0,B-R,0.0,244.542,1.0,0.0
1,B-RT,0.0,244.542,1.0,0.0
2,N-R,0.0,237.774,1.0,0.0
3,N-RT,0.0,237.774,1.0,0.0
4,NFB1-R,0.0,237.998,1.0,0.0
5,P-R,0.0,239.928,1.0,0.0
6,P-RT,0.0,239.928,1.0,0.0
7,S-R,0.0,234.042,1.0,0.0
8,S-RT,0.0,234.042,1.0,0.0
9,SQ-C,0.0,145.488,1.0,0.0


,comparison,criterion,mean,mean_first_half,batch_se,p05,p95,share_positive,n
0,N-R − P-R,P (bottom-quartile attainment 2040),-0.000142,0.000095,0.000136,-0.006791,0.005742,0.448,500
1,S-R − P-R,P (bottom-quartile attainment 2040),-0.028931,-0.028541,0.000286,-0.039437,-0.018857,0.000,500
2,P-RT − P-R,P (bottom-quartile attainment 2040),0.000000,0.000000,0.000000,0.000000,0.000000,0.000,500
3,SQ-C − P-R,P (bottom-quartile attainment 2040),-0.067220,-0.066974,0.000289,-0.079018,-0.053906,0.000,500
4,B-R − P-R,P (bottom-quartile attainment 2040),-0.009467,-0.009243,0.000236,-0.021542,0.001992,0.090,500
